# Phase 5: Expert Aggressive — Toxic-BERT + Hybrid

**Expert Adaptation** strategy to break the F1 plateau:

| Change | Setting |
|--------|--------|
| Base model | `unitary/toxic-bert` (head-only fine-tune) |
| LR bottleneck | TF-IDF `max_features=250` |
| Threshold | Val-set search maximizing **F1-toxic** |
| Hybrid weights | **0.7** Toxic-BERT + **0.3** LR |
| Augmentation | EN→**DE**→EN back-translation (higher diversity) |

Run from repo root (long-running — augmentation + fine-tune):

```bash
uv sync --extra hf --extra train
uv run python -m src.pipeline.run_expert_pipeline
```

Or execute the pipeline cell below inside this notebook.

## 0. Setup

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

cfg_path = PROJECT_ROOT / "configs" / "expert_training.yaml"
cfg = yaml.safe_load(open(cfg_path))
reports_dir = PROJECT_ROOT / "reports" / "expert"
print(f"Config: {cfg_path.name}")
print(f"Pivot lang: {cfg['augmentation']['pivot_lang']}")
print(f"Model: {cfg['transformer']['model_id']} ({cfg['transformer']['freeze_mode']})")

Config: expert_training.yaml
Pivot lang: de
Model: unitary/toxic-bert (head_only)


## 1. Run Phase 5 pipeline

In [2]:
from src.pipeline.run_expert_pipeline import run_expert_pipeline

metrics = run_expert_pipeline(config_path=cfg_path)
run_id = metrics["run_id"]
print(f"Completed run_id={run_id}")

/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-24 19:39:47 | INFO     | src.pipeline.run_expert_pipeline | ============================================================
2026-05-24 19:39:47 | INFO     | src.pipeline.run_expert_pipeline | EXPERT PIPELINE (Phase 5) — run=20260524_193947
2026-05-24 19:39:47 | INFO     | src.pipeline.run_expert_pipeline | ============================================================
2026-05-24 19:39:47 | INFO     | src.data.loader | Cargando dataset: /Users/miraekang/proyectos/ai-nlp/data/raw/youtoxic_english_1000.csv
2026-05-24 19:39:47 | INFO     | src.data.loader |   Shape: (1000, 15)
2026-05-24 19:39:47 | INFO     | src.data.loader |   Columnas validadas ✅
2026-05-24 19:39:47 | WARNING  | src.data.loader |   3 duplicados eliminados
2026-05-24 19:39:47 | INFO     | src.data.loader |   Toxicos: 459 (46.0%)
2026-05-24 19:39:47 | INFO     | src.pipeline.run_expert_pipeline | Augmentation EN→DE→EN (toxic only)
2026-05-24 19:39:47 | INFO     | src.features.augmentation | Back-translation: 312 toxic 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9092.34it/s]


2026-05-24 19:42:08 | INFO     | src.features.augmentation | Dedup: kept 209/295 (dropped 86 with cosine > 0.95)
2026-05-24 19:42:08 | INFO     | src.features.augmentation | Train size after augmentation: 886 (+209)
2026-05-24 19:42:08 | INFO     | src.pipeline.run_expert_pipeline | LR-TFIDF (max_features=250) + gap search
2026-05-24 19:42:08 | INFO     | src.models.hybrid_ensemble | Training stable LR — C=0.05
2026-05-24 19:42:08 | INFO     | src.models.hybrid_ensemble | LR gap search — C=0.05 max_features=250 min_df=3 train_f1=0.7703 test_f1=0.6563 gap=0.1139
2026-05-24 19:42:08 | INFO     | src.models.hybrid_ensemble | Training stable LR — C=0.03
2026-05-24 19:42:08 | INFO     | src.models.hybrid_ensemble | LR gap search — C=0.03 max_features=250 min_df=5 train_f1=0.7572 test_f1=0.6563 gap=0.1008
2026-05-24 19:42:08 | INFO     | src.models.hybrid_ensemble | Training stable LR — C=0.02
2026-05-24 19:42:08 | INFO     | src.models.hybrid_ensemble | LR gap search — C=0.02 max_features=2

Map: 100%|██████████| 200/200 [00:00<00:00, 23323.72 examples/s]
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `6`.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8948.39it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


...

/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Map: 100%|██████████| 886/886 [00:00<00:00, 27925.88 examples/s]
/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


2026-05-24 19:43:51 | INFO     | src.pipeline.run_expert_pipeline | Expert report: /Users/miraekang/proyectos/ai-nlp/reports/expert/integrated_report_20260524_193947.md
2026-05-24 19:43:51 | INFO     | src.pipeline.run_expert_pipeline | ============================================================
2026-05-24 19:43:51 | INFO     | src.pipeline.run_expert_pipeline | Toxic-BERT-expert: F1-toxic=0.7489 ⚠️ | toxic gap=0.0418 ✅ | threshold=0.33
2026-05-24 19:43:51 | INFO     | src.pipeline.run_expert_pipeline | LR-TFIDF-expert: F1-toxic=0.6301 ⚠️ | toxic gap=0.0008 ✅ | threshold=0.05
2026-05-24 19:43:51 | INFO     | src.pipeline.run_expert_pipeline | Hybrid-ToxicBERT+LR: F1-toxic=0.7489 ⚠️ | toxic gap=0.0428 ✅ | threshold=0.38
2026-05-24 19:43:51 | INFO     | src.pipeline.run_expert_pipeline | ============================================================
Completed run_id=20260524_193947


## 2. Holdout test — F1-toxic and gap

In [3]:
def _row(key, label):
    m = metrics.get(key, {})
    if not m:
        return None
    return {
        "model": label,
        "f1_toxic_test": m.get("f1_toxic"),
        "f1_toxic_train": m.get("f1_toxic_train"),
        "toxic_gap_pp": m.get("train_test_gap_toxic_pp"),
        "gap_ok_<5pp": m.get("gap_toxic_ok", False),
        "f1_target_>0.75": (m.get("f1_toxic") or 0) > 0.75,
        "threshold": m.get("threshold"),
        "roc_auc": m.get("roc_auc"),
    }

summary = pd.DataFrame(
    [
        r
        for r in [
            _row("transformer", "Toxic-BERT"),
            _row("logistic_regression", "LR-TFIDF-250"),
            _row("ensemble", "Hybrid 0.7/0.3"),
        ]
        if r
    ]
)
summary

,model,f1_toxic_test,f1_toxic_train,toxic_gap_pp,gap_ok_<5pp,f1_target_>0.75,threshold,roc_auc
0,Toxic-BERT,0.7489,0.7907,4.18,True,False,0.33,0.8768
1,LR-TFIDF-250,0.6301,0.6309,0.08,True,False,0.05,0.7056
2,Hybrid 0.7/0.3,0.7489,0.7917,4.28,True,False,0.38,0.8773


## 3. Integrated report

In [4]:
from IPython.display import Markdown, display

md_path = reports_dir / f"integrated_report_{run_id}.md"
if md_path.exists():
    display(Markdown(md_path.read_text()))
else:
    print("Report not found")

# Phase 5 Expert Adaptation — 20260524_193947

## Targets
- Test **F1-toxic** > 0.75
- |Train F1-toxic − Test F1-toxic| < 5 pp (0.05)

## Holdout test (tuned thresholds on validation)

| Model | F1-toxic (test) | F1-toxic (train) | Toxic gap (pp) | Threshold | Gap OK |
|-------|-------------------|--------------------|----------------|-----------|--------|
| Toxic-BERT | 0.7489 | 0.7907 | 4.18 | 0.33 | ✅ |
| LR-TFIDF (250 feat) | 0.6301 | 0.6309 | 0.08 | 0.05 | ✅ |
| Hybrid 0.7/0.3 | 0.7489 | 0.7917 | 4.28 | 0.38 | ✅ |

## Augmentation
- Pivot language: de
- Train size: 677 → 886 (+209)

## Verdict
**Toxic-BERT** toxic gap < 5 pp ✅; **Hybrid** toxic gap < 5 pp ✅

- JSON: `reports/expert/expert_run_20260524_193947.json`


## Conclusion

Phase 5 applies **Toxic-BERT** with a frozen backbone, a **250-feature** LR safety net, **validation threshold tuning** on F1-toxic, and a **0.7/0.3** hybrid.
Augmentation uses a **German** pivot for more diverse toxic paraphrases.

Success criteria:
- **F1-toxic (test) > 0.75**
- **|F1-toxic train − F1-toxic test| < 5 pp** (`gap_toxic_ok`)

Artifacts: `models/expert_toxic_bert/`, `models/expert_lr_tfidf.joblib`, `reports/expert/expert_run_{run_id}.json`.